# 🚀 SOHO-CL vs FLY-CL: Continual Learning Benchmark trên Kaggle

Notebook này được thiết kế để tự động clone mã nguồn từ GitHub, cài đặt môi trường, và tự động chạy thực nghiệm, đánh giá, vẽ biểu đồ so sánh hai thuật toán **FLY-CL (Baseline)** và **SOHO-CL (Đề xuất)** trên 3 Dataset lớn: `CIFAR-100`, `CUB-200-2011`, và `ImageNet-R`.

In [ ]:
# 1. Khởi tạo môi trường: Clone repo SOHO-CL và cài đặt thư viện
import os

%cd /kaggle/working
!rm -rf SOHO-CL
!git clone https://github.com/ZaPhat206/SOHO-CL.git

%cd /kaggle/working/SOHO-CL
!pip install timm==0.9.16 pandas matplotlib -q

print("✅ Clone Repo và cài đặt môi trường thành công!")

In [ ]:
# 2. Khai báo hàm tự động chạy và lấy Metrics
import subprocess
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

def run_experiment(method, dataset, num_classes, num_tasks=20, coding_level=0.1):
    print(f"\n⏳ Đang chạy {method.upper()} trên {dataset} (Tasks: {num_tasks})...")
    
    # Thư mục gốc chứa Dataset trên Kaggle
    root_path = "/kaggle/input/datasets/zaphat206"
    
    # Tuỳ chỉnh coding_level cho từng method để công bằng
    if method == 'flycl':
        cl = 0.01
    else:
        cl = coding_level

    cmd = [
        "python", "main.py", 
        "--method", method, 
        "--dataset", dataset, 
        "--num_classes", str(num_classes), 
        "--num_tasks", str(num_tasks),
        "--coding_level", str(cl),
        "--root", root_path
    ]
    
    # Đọc luồng output real-time để báo cáo tiến độ thay vì chờ đợi mòn mỏi
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    output_lines = []
    
    for line in iter(process.stdout.readline, ''):
        output_lines.append(line)
        # Chỉ in ra những dòng chứa thông tin quan trọng của Task
        if "[Task" in line or "Starting Continual" in line or "Evaluation Summary" in line:
            print("   " + line.strip())
            
    process.stdout.close()
    process.wait()
    output = "".join(output_lines)
    
    # Phân tích chuỗi output để lấy Metrics
    metrics = {"Method": method.upper(), "Dataset": dataset}
    try:
        metrics["AA (%)"] = float(output.split("Accumulated Accuracy\n")[1].split("\n")[0])
        metrics["LA (%)"] = float(output.split("Learning Accuracy (LA): ")[1].split("\n")[0])
        metrics["Forgetting (%)"] = float(output.split("Forgetting (F): ")[1].split("%")[0])
        metrics["BWT (%)"] = float(output.split("Backward Transfer (BWT): ")[1].split("%")[0])
        metrics["Memory (MB)"] = float(output.split("Memory Footprint (excluding frozen backbone): ")[1].split(" MB")[0])
        metrics["Avg Train Time (s)"] = float(output.split("Average Training Time\n")[1].split("\n")[0])
        
        # Trích xuất đoạn văn bản chứa ma trận độ chính xác
        matrix_str = output.split("Accuracy Matrix\n")[1].split("\nAverage Accuracy")[0]
        metrics["_matrix_str"] = matrix_str
        print(f"✅ HOÀN TẤT! AA: {metrics['AA (%)']} | Forgetting: {metrics['Forgetting (%)']}\n")
    except Exception as e:
        print(f"❌ Lỗi khi phân tích output của {method} trên {dataset}!")
        print(output[-1000:])
        
    return metrics

In [ ]:
# 3. Khởi chạy chuỗi thực nghiệm toàn diện
experiments = [
    {"dataset": "CIFAR-100", "num_classes": 100},
    {"dataset": "CUB-200-2011", "num_classes": 200},
    {"dataset": "ImageNet-R", "num_classes": 200}
]

methods = ['flycl', 'sohocl']
results = []

for exp in experiments:
    for m in methods:
        res = run_experiment(m, exp['dataset'], exp['num_classes'])
        if "AA (%)" in res:
            results.append(res)

# Hiển thị bảng tóm tắt kết quả siêu gọn gàng
df_results = pd.DataFrame(results)
display_df = df_results.drop(columns=["_matrix_str"])
display(Markdown("### 📊 Bảng Tổng Hợp Kết Quả Thực Nghiệm"))
display(display_df)

In [ ]:
# 4. Render đồ thị Parabol tự động
import ast
from notebooks.plot_results import compute_metrics, plot_comparisons

def extract_matrix(matrix_str):
    lines = matrix_str.strip().split('\n')
    matrix = []
    for line in lines:
        clean_line = line.replace("'0.00'", "0.0")
        matrix.append(ast.literal_eval(clean_line))
    return matrix

for exp in experiments:
    dataset = exp['dataset']
    fly_data = next((r for r in results if r['Dataset'] == dataset and r['Method'] == 'FLYCL'), None)
    soho_data = next((r for r in results if r['Dataset'] == dataset and r['Method'] == 'SOHOCL'), None)
    
    if fly_data and soho_data:
        m_fly = extract_matrix(fly_data['_matrix_str'])
        m_soho = extract_matrix(soho_data['_matrix_str'])
        print(f"\n" + "="*60)
        print(f"📈 ĐỒ THỊ SO SÁNH TRÊN DATASET: {dataset}")
        print("="*60)
        plot_comparisons(m_fly, m_soho, dataset_name=dataset)